# Documenation
- Program: cocoClass.py
- Programmer/s: Cristina C. Villasor
- Date Written: June 15, 2025
- Last Revised: December 18, 2025
- Purpose: Train Faster R-CNN model

# Install dependencies
- For training, we need detectron2. Detectro2 is a framework created by Meta, known for its various tool for computer vision jobs.

In [ ]:
# Detectron
import sys, os, distutils.core
from IPython.display import clear_output

!python -m pip install pyyaml==5.1
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

clear_output(wait=True)
print("Detectron loaded successfully")

In [ ]:
!pip install gdown roboflow

clear_output(wait=True)
print("Installed successfully")

In [ ]:
# Importing dependencies
import torch, detectron2
from detectron2.utils.logger import setup_logger
setup_logger()
print("detectron2 version:", detectron2.__version__)

import numpy as np
import os, json, cv2, random
import PIL
import matplotlib.pyplot as plt
import seaborn as sns
import logging
import pickle

from roboflow import Roboflow
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
from tqdm import tqdm
import warnings

logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

# Get Dataset

In [ ]:
rf = Roboflow(api_key="Zykmbeiu1pSoB9Tzeom8")
project = rf.workspace("thesis-vrpuo").project("lspd-hod-ozsk1")
version = project.version(2)
dataset = version.download("coco")

clear_output(wait=True)
print("Data loaded successfully")

# Register Dataset

In [ ]:
# Clean
register_coco_instances(
    "train",
    {},
    "/kaggle/working/lspd+hod-2/train/_annotations.coco.json",
    "/kaggle/working/lspd+hod-2/train/"
)
register_coco_instances(
    "valid",
    {},
    "/kaggle/working/lspd+hod-2/valid/_annotations.coco.json",
    "/kaggle/working/lspd+hod-2/valid/"
)
register_coco_instances(
    "test",
    {},
    "/kaggle/working/lspd+hod-2/test/_annotations.coco.json",
    "/kaggle/working/lspd+hod-2/test/"
)

# Check sample

In [ ]:
def visualize_samples(dataset):
    dataset_metadata = MetadataCatalog.get(dataset)
    dataset_dicts = DatasetCatalog.get(dataset)

    for data in random.sample(dataset_dicts, 2):
        img = cv2.imread(data["file_name"])
        visualizer = Visualizer(img[:, :, ::-1], metadata = dataset_metadata, scale = 1.2)
        visualize = visualizer.draw_dataset_dict(data)

        plt.figure(figsize = (12, 8))
        plt.imshow(cv2.cvtColor(visualize.get_image()[:, :, ::-1], cv2.COLOR_BGR2RGB))

In [ ]:
visualize_samples("train")

In [ ]:
# classes 
classes = []
with open("/kaggle/working/lspd+hod-2/train/_annotations.coco.json", 'r') as f:
    for line in f:
        classes.append(json.loads(line))
        
classes[0]['categories']

# Train
- This section is where training happens
- Our training is continuous, as long as its performance is increasing.

In [ ]:
EPOCHS = 100000 # 70000
NUM_CLASSES = 11
BASE_LR = 0.0002 # 0.002

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))

cfg.DATASETS.TRAIN = ("train",)
cfg.DATASETS.TEST = ("valid",)
cfg.DATALOADER.NUM_WORKERS = 4

cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")  

cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.GAMMA = 0.1
cfg.SOLVER.BASE_LR = BASE_LR
cfg.SOLVER.STEPS = (80000, 90000)
cfg.SOLVER.MAX_ITER = EPOCHS
cfg.SOLVER.WEIGHT_DECAY = 0.0001

cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512
cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = 0.5

cfg.INPUT.MIN_SIZE_TRAIN = (640, 672, 704, 736, 768, 800)
cfg.INPUT.MAX_SIZE_TRAIN = 1333
cfg.TEST.AUG.ENABLED = True

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# uncomment below to train
trainer = DefaultTrainer(cfg) 
trainer.resume_or_load(resume=True)
trainer.train()

## Save Model
- The **cfg** file contains the architecture of the model, data processing, dataset info and the parameters used for training. 
- Without this, you won't be able to use the model correctly

In [ ]:
cfg.MODEL.WEIGHTS = "/kaggle/working/output/model_final.pth" 
# cfg.MODEL.WEIGHTS = "/kaggle/input/model-final-v3/pytorch/default/1/model_final.pth" 
predictor = DefaultPredictor(cfg)

with open("cfg.pkl", "wb") as f:
    pickle.dump(cfg, f)

## Performance of Trained Model
- This section will show how the total loss, accuracy, false negative, and low box regression
- # original code from https://eidos-ai.medium.com/training-on-detectron2-with-a-validation-set-and-plot-loss-on-it-to-avoid-overfitting-6449418fbf4e

In [ ]:
def reading_metrics_from_json(metrics_path):
    metrics = []
    
    with open(metrics_path, 'r') as f:
        for line in f:
            metrics.append(json.loads(line))
    return metrics

In [ ]:
def metric(metrics, specific_metric):
    metric = []
    for i in range(len(metrics)):
        try:
            metric.append(metrics[i][specific_metric])
        except KeyError:
            pass

    return metric

In [ ]:
train_metrics = reading_metrics_from_json('/kaggle/working/output/metrics.json')
total_loss = metric(train_metrics, 'total_loss')
cls_acc = metric(train_metrics, 'fast_rcnn/cls_accuracy')
false_neg = metric(train_metrics, 'fast_rcnn/false_negative')
loss_box_reg = metric(train_metrics, 'loss_box_reg')
iters = np.arange(1,EPOCHS,EPOCHS/len(total_loss))

In [ ]:
fig, axs = plt.subplots(1,4, figsize = (17, 4), dpi = 120)

axs[0].grid(linestyle = 'dashdot')
axs[0].plot(iters, loss_box_reg)
axs[0].set_xlabel('epochs', fontsize = 10)
axs[0].set_title('Loss Box Regression', fontsize = 10)
tit0 = ' (the last value {0:.4f})'.format(loss_box_reg[-1])
axs[0].set_title('Loss Box Regression ' + tit0, fontsize = 10, color = 'red')

axs[1].grid(linestyle = 'dashdot')
axs[1].plot(iters, cls_acc)
axs[1].set_xlabel('epochs', fontsize = 10)
tit1 = ' (the last value {0:.3f})'.format(cls_acc[-1])
axs[1].set_title('Class Accuracy ' + tit1, fontsize = 10, color = 'red')

axs[2].grid(linestyle = 'dashdot')
axs[2].plot(iters, total_loss)
axs[2].set_xlabel('epochs', fontsize = 10)
tit2 = ' (the last value {0:.4f})'.format(total_loss[-1])
axs[2].set_title('Total Loss ' + tit2, fontsize = 10, color = 'red')

axs[3].grid(linestyle = 'dashdot')
axs[3].plot(iters, false_neg)
axs[3].set_xlabel('epochs', fontsize = 10)
axs[3].set_title('False Negative', fontsize = 10, color = 'red')

In [ ]:
evaluator = COCOEvaluator("valid", cfg, False, output_dir="./output/")
val_loader = build_detection_test_loader(cfg, "valid")

metrics = inference_on_dataset(trainer.model, val_loader, evaluator)

# Inference

In [ ]:
# Ground Truth and Prediction
def inference(dataset):
    fig, axs = plt.subplots(2, 2, figsize=(12, 8))
    
    dataset_metadata = MetadataCatalog.get(dataset)
    dataset_dicts = DatasetCatalog.get(dataset)

    samples = random.sample(dataset_dicts, 2)
    
    for i, data in enumerate(samples):
        img = cv2.imread(data["file_name"])
        
        visualizer_truth = Visualizer(img[:, :, ::-1], metadata = dataset_metadata, scale = 1.2)
        visualize_truth = visualizer_truth.draw_dataset_dict(data)

        outputs = predictor(img)
        visualizer_pred = Visualizer(img[:, :, ::-1], metadata = dataset_metadata, scale = 1.2)
        visualize_pred = visualizer_pred.draw_instance_predictions(outputs["instances"].to("cpu"))

        axs[i, 0].imshow(cv2.cvtColor(visualize_truth.get_image()[:, :, ::-1], cv2.COLOR_BGR2RGB))
        axs[i, 0].set_title("Ground Truth")
        axs[i, 0].axis('off')
        
        axs[i, 1].imshow(cv2.cvtColor(visualize_pred.get_image()[:, :, ::-1], cv2.COLOR_BGR2RGB))
        axs[i, 1].set_title("Prediction")
        axs[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

## Load model

In [ ]:
# Load the config from the pickle file
with open("/kaggle/working/cfg.pkl", "rb") as f:
    cfg = pickle.load(f)

# Set the weights for the Faster R-CNN model
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = "/kaggle/working/output/model_final.pth"

cfg.MODEL.ROI_HEADS.NUM_CLASSES = 11
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

# Initialize the predictor
predictor = DefaultPredictor(cfg)

In [ ]:
# Test Dataset
evaluator = COCOEvaluator("test", cfg, False, output_dir="./output/")
test_loader = build_detection_test_loader(cfg, "test")  # test dataset

inference_on_dataset(predictor.model, test_loader, evaluator)

inference("test")

# Performance Metrics
- This section computes for the iou to get confusion matrix (TP, TN, FP, FN)
    - **True Positive (TP)**: correct predictions
    - **True Negative (TN)**: correct predictions that no images must be detected. But in object detection this is irrelevant ([Padilla et al., 2020](https://www.researchgate.net/publication/343194514_A_Survey_on_Performance_Metrics_for_Object-Detection_Algorithms)), if included it might detect background as TN, hence making the calculation unreliable. 
    - **False Positive (FP)**: predicted objects wrong
    - **Flase Negative (FN)**: fails to detect objects

In [ ]:
def calculate_iou(box1, box2):
    """Calculate IoU between two boxes in XYWH format
    Args:
        box1 (list): predicted boxes containing x/y position, width, and height
        box2 (list): ground truth boxes containing x/y position, width, and height
    Returns:
        iou (float): IoU score
    """
    # Convert to xyxy format
    box1_xyxy = [box1[0], box1[1], box1[0] + box1[2], box1[1] + box1[3]]
    box2_xyxy = [box2[0], box2[1], box2[0] + box2[2], box2[1] + box2[3]]

    # Calculate intersection
    x1 = max(box1_xyxy[0], box2_xyxy[0])
    y1 = max(box1_xyxy[1], box2_xyxy[1])
    x2 = min(box1_xyxy[2], box2_xyxy[2])
    y2 = min(box1_xyxy[3], box2_xyxy[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)

    # Calculate union
    box1_area = box1[2] * box1[3]
    box2_area = box2[2] * box2[3]
    union = box1_area + box2_area - intersection
    
    return intersection / union if union > 0 else 0

In [ ]:
def evaluate_detections(predictor, dataset_name, iou_threshold, score_threshold):
    """Evaluate object detection results and calculate confusion matrix metrics
    Args:
        predictor (DefaultPredictor): Detectron2 predictor object for inference
        dataset_name (str): Name of the registered dataset to evaluate
        iou_threshold (float): IoU threshold to determine true positive (e.g., 0.5)
        score_threshold (float): Confidence score threshold for predictions (e.g., 0.5)
    Returns:
        tp (int): True positives
        tn (int): True negatives
        fp (int): False positives
        fn (int): False negatives
        accuracy (float): Detection accuracy
        precision (float): Precision score
        recall (float): Recall score
        f1 (float): F1 score
    """
    dataset_dicts = DatasetCatalog.get(dataset_name)

    total_gt, tp, tn, fp, fn = 0, 0, 0, 0, 0

    all_conf = []
    all_tp = []
    
    print("Evaluating on dataset:", dataset_name)

    for d in tqdm(dataset_dicts):
        # Get ground truth boxes
        gt_boxes = [ann["bbox"] for ann in d["annotations"]]
        total_gt += len(gt_boxes)

        # Get predictions
        img = cv2.imread(d["file_name"])
        outputs = predictor(img)
        pred_boxes = outputs["instances"].pred_boxes.tensor.cpu().numpy()
        scores = outputs["instances"].scores.cpu().numpy()

        # Filter predictions by confidence threshold
        mask = scores >= score_threshold
        pred_boxes = pred_boxes[mask]
        scores = scores[mask]
        
        # Match predictions to ground truth
        matched_gt = set()

        for pred_idx, pred_box in enumerate(pred_boxes):
            best_iou = 0
            best_gt_idx = -1

            # Convert pred_box from xyxy to xywh format
            pred_box_xywh = [
                pred_box[0],
                pred_box[1],
                pred_box[2] - pred_box[0],
                pred_box[3] - pred_box[1]
            ]

            # Find best matching ground truth box
            for gt_idx, gt_box in enumerate(gt_boxes):
                if gt_idx in matched_gt:
                    continue
                    
                iou = calculate_iou(pred_box_xywh, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx

            # Store confidence and whether it's a true positive
            all_conf.append(scores[pred_idx])
            
            if best_iou >= iou_threshold:
                tp += 1
                matched_gt.add(best_gt_idx)
                all_tp.append(1)
            else:
                fp += 1
                all_tp.append(0)

        # Count unmatched ground truth boxes as false negatives
        fn += len(gt_boxes) - len(matched_gt)
    
    # Calculate precision, recall, and F1 score
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0

    # Print metrics
    print("\nDetection Metrics:")
    print(f"True Positives (TP): {tp}")
    print(f"True Negatives (TN): {tn}")
    print(f"False Positives (FP): {fp}")
    print(f"False Negatives (FN): {fn}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

    return tp, tn, fp, fn, accuracy, precision, recall, f1

In [ ]:
# Confusion Matrix for test set
tp, tn, fp, fn, accuracy, precision, recall, f1 = evaluate_detections(
                    predictor, 
                    "test", 
                    iou_threshold=0.5, 
                    score_threshold=0.7)

# Visualization
- This section shows the visualization of performance metrics of object detection

In [ ]:
# Create confusion matrix visualization
plt.figure(figsize=(10, 8))
confusion_matrix = np.array([[tn, fp], [fn, tp]])  # TN is always 0 in object detection
sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Negative', 'Positive'],
    yticklabels=['Negative', 'Positive'])
plt.title(f'Confusion Matrix (IoU >= {iou_threshold}, Conf >= {score_threshold})')
plt.ylabel('Ground Truth')
plt.xlabel('Predicted')
plt.show()
plt.close()

In [ ]:
# ----- Test -----
labels = ["Accuracy", "Precision", "Recall", "F1 Score"]
perf_metrics = [accuracy, precision, recall, f1]

# Plot bars
plt.bar(labels, perf_metrics, color='skyblue')

# Labels and title
plt.xlabel("Metrics")
plt.ylabel("Values")
plt.title("Performance Metrics")
plt.ylim(0, 1.0)

plt.show()